<a href="https://colab.research.google.com/github/syltaer-utp/s100-tareas/blob/main/S100_Equipo2_Parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S100 — Adelanto de modelado
## Baseline + árbol de regresión  
### Gasto en salud y esperanza de vida (América + Europa, 2000–2024)

**Equipo 2**

Este cuaderno cumple el adelanto del viernes 18:
- Baseline (predecir la media)
- Un modelo supervisado (árbol de regresión)
- Tabla comparativa de métricas (RMSE, MAE, R²)

## 1. Librerías

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Carga del dataset

Se carga `health_panel.csv` (panel país-año de World Bank, WHO y OECD).

In [2]:
DATA_PATH = "health_panel.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape original: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Años: {df['year'].min()} – {df['year'].max()}")
print(f"Países/entidades: {df['country_name'].nunique()}")

Shape original: 17,210 filas × 32 columnas
Años: 1960 – 2024
Países/entidades: 265


## 3. Construcción del subset

1. Solo países de América y Europa (individuales).
2. Solo años ≥ 2000.
3. Solo filas con esperanza de vida y gasto en salud no nulos (sin eso no se puede plantear la hipótesis).
4. Se elimina columnas vacías, redundantes y derivadas.
5. Se crea la variable `continent`.

In [3]:
americas = [
    "Antigua and Barbuda", "Argentina", "Bahamas, The", "Barbados", "Belize",
    "Bolivia", "Brazil", "Canada", "Chile", "Colombia", "Costa Rica", "Cuba",
    "Dominica", "Dominican Republic", "Ecuador", "El Salvador", "Grenada",
    "Guatemala", "Guyana", "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "Paraguay", "Peru", "St. Kitts and Nevis",
    "St. Lucia", "St. Vincent and the Grenadines", "Suriname",
    "Trinidad and Tobago", "United States", "Uruguay", "Venezuela, RB",
]

europe = [
    "Albania", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Greece", "Hungary", "Iceland",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Moldova", "Montenegro", "Netherlands", "North Macedonia", "Norway",
    "Poland", "Portugal", "Romania", "Russian Federation", "Serbia",
    "Slovak Republic", "Slovenia", "Spain", "Sweden", "Switzerland",
    "Ukraine", "United Kingdom",
]

# Filtrar países y crear continente
df_ae = df[df["country_name"].isin(americas + europe)].copy()
df_ae["continent"] = np.where(df_ae["country_name"].isin(americas), "Americas", "Europe")

# Años >= 2000
df_ae = df_ae[df_ae["year"] >= 2000].copy()

# Quitar columnas vacías, redundantes y derivadas
cols_eliminar = [
    "region", "income_group", "iso2_code", "country_code",
    "le_per_gdp_point", "le_per_1k_spend", "le_spend_residual", "efficiency_score",
]
df_ae = df_ae.drop(columns=cols_eliminar, errors="ignore")

# Subset final: LE y gasto no nulos
df_subset = df_ae.dropna(
    subset=["life_expectancy_total", "health_spend_per_capita_usd"]
).copy()

print(f"df_subset: {df_subset.shape[0]:,} filas × {df_subset.shape[1]} columnas")
print(f"Años: {df_subset['year'].min()} – {df_subset['year'].max()}")
print(f"Países: {df_subset['country_name'].nunique()}")
print("\nPor continente:")
print(df_subset["continent"].value_counts())

df_subset: 1,786 filas × 25 columnas
Años: 2000 – 2024
Países: 75

Por continente:
continent
Europe      964
Americas    822
Name: count, dtype: int64
